# Tonkid V-Code Validator

**วิธีใช้:**
1. Run Cell 1 (โค้ดตรวจสอบ) — กด ▶ หรือ Shift+Enter
2. Run Cell 2 — เลือกโหมด:
   - **ตรวจทีละรหัส**: พิมพ์รหัสยืนยันในช่อง
   - **ตรวจจาก CSV**: อัปโหลดไฟล์ CSV แล้วดาวน์โหลดผลลัพธ์

In [ ]:
#@title Cell 1: โค้ดตรวจสอบ (Run ครั้งเดียว)
import re
import csv
from dataclasses import dataclass

# ตารางค่าลับ — LOOKUP_TABLE[คะแนน] = [เกณฑ์1, เกณฑ์2, เกณฑ์3, เกณฑ์4, เกณฑ์5]
LOOKUP_TABLE = {
    1: [3, 7, 2, 9, 5],
    2: [8, 1, 6, 4, 9],
    3: [5, 9, 8, 2, 3],
    4: [1, 4, 3, 7, 6],
}
CRITERIA_NAMES = ["วิเคราะห์ประเด็น", "หลักฐาน", "มุมมอง", "เชื่อมโยง", "สรุป"]
CODE_PATTERN = re.compile(r"^TK-(\d{4})-(\d{5})-(\d+)R-(\d+)C-V(\d{2})-D26$")


@dataclass
class VerificationResult:
    code: str
    valid: bool
    student_id: str = ""
    scores: list = None
    total_score: int = 0
    rounds: int = 0
    copy_count: int = 0
    v_code_given: int = 0
    v_code_expected: int = 0
    error: str = ""


def calculate_v_code(scores, id_digits, rounds):
    lookup_sum = sum(LOOKUP_TABLE[s][i] for i, s in enumerate(scores))
    id_sum = sum(int(d) for d in id_digits)
    raw = lookup_sum + id_sum + rounds
    return raw % 100


def parse_and_verify(code, skip_vcode=False):
    code = code.strip()
    match = CODE_PATTERN.match(code)
    if not match:
        return VerificationResult(code=code, valid=False,
            error="รูปแบบรหัสไม่ถูกต้อง (ต้องเป็น TK-XXXX-XXXXX-XR-XC-VXX-D26)")

    student_id = match.group(1)
    scores = [int(c) for c in match.group(2)]
    rounds = int(match.group(3))
    copy_count = int(match.group(4))
    v_given = int(match.group(5))

    for i, s in enumerate(scores):
        if s < 1 or s > 4:
            return VerificationResult(code=code, valid=False,
                error=f"คะแนนเกณฑ์ที่ {i+1} ({CRITERIA_NAMES[i]}) = {s} ไม่อยู่ในช่วง 1-4")

    if rounds not in (5, 6):
        return VerificationResult(code=code, valid=False,
            error=f"จำนวนรอบ = {rounds} ไม่ปกติ (ควรเป็น 5 หรือ 6)")

    v_expected = calculate_v_code(scores, student_id, rounds)

    if skip_vcode:
        return VerificationResult(
            code=code, valid=True, student_id=student_id,
            scores=scores, total_score=sum(scores), rounds=rounds,
            copy_count=copy_count, v_code_given=v_given, v_code_expected=v_expected,
            error="ข้าม V code (สูตรเก่า)")

    is_valid = (v_given == v_expected)
    return VerificationResult(
        code=code, valid=is_valid, student_id=student_id,
        scores=scores, total_score=sum(scores), rounds=rounds,
        copy_count=copy_count, v_code_given=v_given, v_code_expected=v_expected,
        error="" if is_valid else f"V code ไม่ตรง: ได้ V{v_given:02d} แต่ควรเป็น V{v_expected:02d} → อาจเป็นรหัสปลอม!")


def open_csv_auto(path):
    """เปิด CSV อัตโนมัติ — ลอง encoding หลายตัวจนกว่าจะสำเร็จ"""
    for enc in ["utf-8-sig", "utf-8", "cp874", "latin-1"]:
        try:
            f = open(path, "r", encoding=enc)
            f.readline()
            f.seek(0)
            print(f"  encoding: {enc}")
            return f
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise UnicodeDecodeError(f"ไม่สามารถอ่านไฟล์ได้ — ลอง utf-8-sig, utf-8, cp874, latin-1 แล้วไม่ผ่าน")


def process_csv_colab(input_path, output_path="results.csv", skip_vcode=False):
    results = []
    with open_csv_auto(input_path) as f:
        reader = csv.reader(f)
        header = next(reader, None)

        code_col = None
        if header:
            for i, h in enumerate(header):
                if "TK-" in h or "รหัส" in h.lower() or "code" in h.lower() or "verification" in h.lower():
                    code_col = i
                    break
        if code_col is None and header:
            for i, val in enumerate(header):
                if val.startswith("TK-"):
                    code_col = i
                    results.append(parse_and_verify(val, skip_vcode))
                    break
        if code_col is None:
            code_col = 0

        for row in reader:
            if len(row) > code_col:
                cell = row[code_col].strip()
                if cell.startswith("TK-"):
                    results.append(parse_and_verify(cell, skip_vcode))

    valid_count = sum(1 for r in results if r.valid)
    invalid_count = sum(1 for r in results if not r.valid)

    print(f"\n{'='*60}")
    print(f"  ผลการตรวจสอบ: {len(results)} รหัส")
    print(f"  ✅ ผ่าน: {valid_count}")
    print(f"  ❌ ไม่ผ่าน: {invalid_count}")
    if skip_vcode:
        print(f"  ⚠️ โหมดข้าม V code (ไม่ตรวจ checksum)")
    print(f"{'='*60}\n")

    if invalid_count > 0:
        print("รหัสที่ไม่ผ่าน:")
        for r in results:
            if not r.valid:
                print(f"  {r.code} — ❌ — {r.error}")
        print()

    with open(output_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "รหัสยืนยัน", "สถานะ", "รหัส_นศ",
            "วิเคราะห์ประเด็น", "หลักฐาน", "มุมมอง", "เชื่อมโยง", "สรุป",
            "คะแนนรวม", "จำนวนรอบ", "การคัดลอก",
            "V_ที่ได้", "V_ที่ถูกต้อง", "หมายเหตุ"
        ])
        for r in results:
            scores = r.scores or [0, 0, 0, 0, 0]
            if r.valid and r.error == "ข้าม V code (สูตรเก่า)":
                status_text = "ผ่าน (ข้าม V)"
            elif r.valid:
                status_text = "ถูกต้อง"
            else:
                status_text = "ไม่ถูกต้อง"
            writer.writerow([
                r.code, status_text,
                r.student_id, *scores,
                r.total_score, r.rounds, r.copy_count,
                f"V{r.v_code_given:02d}" if r.student_id else "",
                f"V{r.v_code_expected:02d}" if r.student_id else "",
                r.error
            ])

    print(f"📁 บันทึกผลลัพธ์ที่: {output_path}")
    return results


print("✅ โหลดโค้ดตรวจสอบเรียบร้อยแล้ว — ไป Cell 2 ได้เลย!")

In [ ]:
#@title Cell 2: เลือกโหมดการใช้งาน
mode = "CSV"  #@param ["ตรวจทีละรหัส", "CSV"]
vcode_mode = "ข้าม V code (รอบเก่า)"  #@param ["ตรวจ V code (สูตรใหม่)", "ข้าม V code (รอบเก่า)"]
skip_vcode = (vcode_mode == "ข้าม V code (รอบเก่า)")

if skip_vcode:
    print("⚠️ โหมดข้าม V code — ดึงคะแนน/ID/รอบจากรหัส โดยไม่ตรวจ V code")
    print()

if mode == "ตรวจทีละรหัส":
    code = input("พิมพ์รหัสยืนยัน: ").strip()
    if code:
        r = parse_and_verify(code, skip_vcode=skip_vcode)
        if r.valid and r.error == "ข้าม V code (สูตรเก่า)":
            status = "✅ ผ่าน (ข้าม V)"
        elif r.valid:
            status = "✅ ถูกต้อง"
        else:
            status = "❌ ไม่ถูกต้อง"
        print(f"\nสถานะ: {status}")
        if r.student_id:
            print(f"รหัส นศ. 4 ตัวท้าย: {r.student_id}")
            for name, score in zip(CRITERIA_NAMES, r.scores):
                print(f"  {name}: {score}")
            print(f"คะแนนรวม: {r.total_score}/20")
            print(f"จำนวนรอบ: {r.rounds} {' (มีทบทวน)' if r.rounds == 6 else ''}")
            print(f"การคัดลอก: {r.copy_count} ครั้ง")
            if not skip_vcode:
                print(f"V code: V{r.v_code_given:02d} (คาดหวัง: V{r.v_code_expected:02d})")
        if r.error and r.error != "ข้าม V code (สูตรเก่า)":
            print(f"\n⚠️ {r.error}")

elif mode == "CSV":
    from google.colab import files
    print("เลือกไฟล์ CSV ที่มีรหัสยืนยัน...")
    uploaded = files.upload()
    input_file = list(uploaded.keys())[0]
    print(f"\nได้รับไฟล์: {input_file}")

    results = process_csv_colab(input_file, "results.csv", skip_vcode=skip_vcode)

    print("\nกำลังดาวน์โหลด results.csv ...")
    files.download("results.csv")